# 01 — CNN embedding extraction (cached)

One-time extraction of frozen ImageNet features for Tasks 1 and 2:

- **ResNet18** — 512-dim (`model.fc = Identity()`)
- **EfficientNet-B0** — 1280-dim (`model.classifier = Identity()`)

Preprocessing: resize to 224×224, ImageNet normalisation. Outputs are written to
`outputs/embeddings/` as `.npy` arrays plus `_ids.json` sidecars (row order = metadata order).

Run once before `02_task1_models.ipynb` and `03_task2_models.ipynb`.

**Kernel:** select **Python (MLAssignment2 .venv)** (project `.venv` with `torch`). If missing, run from repo root:
`python3 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt && python -m ipykernel install --user --name mlassignment2 --display-name "Python (MLAssignment2 .venv)"`

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    EfficientNet_B0_Weights,
    ResNet18_Weights,
    efficientnet_b0,
    resnet18,
)
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from utils import (
    EMBEDDINGS_DIR,
    REPO_ROOT as UTILS_ROOT,
    embedding_cache_paths,
    load_task,
    save_embedding_cache,
    seed_everything,
)

assert REPO_ROOT == UTILS_ROOT

SEED = 42
BATCH_SIZE = 64
NUM_WORKERS = 0  # increase on Linux if desired; 0 is safest on macOS notebooks

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Embeddings dir: {EMBEDDINGS_DIR}")

Device: cpu
Embeddings dir: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings


In [2]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def build_backbone(name: str) -> tuple[nn.Module, int]:
    """Return (eval-mode frozen backbone, embedding dimension)."""
    if name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Identity()
        dim = 512
    elif name == "effnetb0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        model.classifier = nn.Identity()
        dim = 1280
    else:
        raise ValueError(f"Unknown backbone: {name}")
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model.to(device), dim


class MetaImageDataset(Dataset):
    def __init__(self, meta_df, root: Path, transform):
        self.paths = [root / p for p in meta_df["image_path"].tolist()]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        from PIL import Image

        with Image.open(self.paths[idx]) as im:
            img = im.convert("RGB")
        return self.transform(img)


@torch.no_grad()
def extract_embeddings(meta_df, task_dir: Path, model: nn.Module, dim: int) -> np.ndarray:
    dataset = MetaImageDataset(meta_df, task_dir, transform)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda",
    )
    chunks = []
    for batch in tqdm(loader, desc=task_dir.name, leave=False):
        batch = batch.to(device)
        out = model(batch)
        if out.dim() > 2:
            out = torch.flatten(out, 1)
        chunks.append(out.cpu().numpy())
    emb = np.concatenate(chunks, axis=0).astype(np.float32)
    assert emb.shape == (len(meta_df), dim), (emb.shape, len(meta_df), dim)
    return emb

In [3]:
BACKBONES = ["resnet18", "effnetb0"]
TASKS = [1, 2]

summary = []

for task in TASKS:
    bundle = load_task(task)
    task_dir = bundle.task_dir

    for backbone in BACKBONES:
        model, dim = build_backbone(backbone)
        print(f"\n=== Task {task} | {backbone} ({dim}-d) ===")

        for split, meta, ids in [
            ("train", bundle.train_meta, bundle.train_ids),
            ("test", bundle.test_meta, bundle.test_ids),
        ]:
            npy_path, json_path = embedding_cache_paths(task, split, backbone)
            if npy_path.exists() and json_path.exists():
                emb = np.load(npy_path)
                with open(json_path) as f:
                    cached_ids = json.load(f)["image_ids"]
                if list(cached_ids) == list(ids) and emb.shape == (len(ids), dim):
                    print(f"  {split}: cache hit {npy_path.name} {emb.shape}")
                    summary.append((task, split, backbone, emb.shape, "cached"))
                    continue
                print(f"  {split}: stale cache — re-extracting")

            emb = extract_embeddings(meta, task_dir, model, dim)
            save_embedding_cache(emb, ids, task, split, backbone)
            print(f"  {split}: saved {npy_path.name} {emb.shape}")
            summary.append((task, split, backbone, emb.shape, "written"))

        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

print("\n--- Summary ---")
for row in summary:
    print(row)


=== Task 1 | resnet18 (512-d) ===
  train: cache hit task1_train_resnet18.npy (3750, 512)
  test: cache hit task1_test_resnet18.npy (1250, 512)

=== Task 1 | effnetb0 (1280-d) ===
  train: cache hit task1_train_effnetb0.npy (3750, 1280)
  test: cache hit task1_test_effnetb0.npy (1250, 1280)

=== Task 2 | resnet18 (512-d) ===
  train: cache hit task2_train_resnet18.npy (417, 512)
  test: cache hit task2_test_resnet18.npy (180, 512)

=== Task 2 | effnetb0 (1280-d) ===
  train: cache hit task2_train_effnetb0.npy (417, 1280)
  test: cache hit task2_test_effnetb0.npy (180, 1280)

--- Summary ---
(1, 'train', 'resnet18', (3750, 512), 'cached')
(1, 'test', 'resnet18', (1250, 512), 'cached')
(1, 'train', 'effnetb0', (3750, 1280), 'cached')
(1, 'test', 'effnetb0', (1250, 1280), 'cached')
(2, 'train', 'resnet18', (417, 512), 'cached')
(2, 'test', 'resnet18', (180, 512), 'cached')
(2, 'train', 'effnetb0', (417, 1280), 'cached')
(2, 'test', 'effnetb0', (180, 1280), 'cached')


In [4]:
# Sanity: reload caches and assert alignment with metadata
from utils import load_embedding_cache

for task in TASKS:
    bundle = load_task(task)
    for backbone in BACKBONES:
        for split, expected_ids in [("train", bundle.train_ids), ("test", bundle.test_ids)]:
            emb, ids = load_embedding_cache(task, split, backbone)
            assert list(ids) == list(expected_ids)
            expected_dim = 512 if backbone == "resnet18" else 1280
            assert emb.shape == (len(expected_ids), expected_dim)
            assert np.isfinite(emb).all()
    print(f"Task {task}: all embedding caches OK")

sorted(EMBEDDINGS_DIR.glob("task*"))

Task 1: all embedding caches OK
Task 2: all embedding caches OK


[PosixPath('/Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings/task1_test_effnetb0.npy'),
 PosixPath('/Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings/task1_test_effnetb0_ids.json'),
 PosixPath('/Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings/task1_test_resnet18.npy'),
 PosixPath('/Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings/task1_test_resnet18_ids.json'),
 PosixPath('/Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings/task1_train_effnetb0.npy'),
 PosixPath('/Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings/task1_train_effnetb0_ids.json'),
 PosixPath('/Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings/task1_train_resnet18.npy'),
 PosixPath('/Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings/task1_train_resnet18_ids.json'),
 PosixPath('/Users/skandaramanan/Documents/ML/MLAssignment2/outputs/embeddings/task2_test_effnetb0.npy'),
 PosixPath('/Users/ska